In [ ]:
%%writefile reduction.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>

#define N 1024

// ---------------- GPU KERNEL ----------------
__global__ void reduceSum(float *input, float *output) {
    __shared__ float shared[1024];

    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;

    // Load into shared memory
    shared[tid] = input[i];
    __syncthreads();

    // Reduction
    for(int stride = blockDim.x/2; stride > 0; stride /= 2) {
        if(tid < stride) {
            shared[tid] += shared[tid + stride];
        }
        __syncthreads();
    }

    // Store result
    if(tid == 0) {
        output[blockIdx.x] = shared[0];
    }
}

// ---------------- MAIN ----------------
int main() {
    int size = N * sizeof(float);

    float *h_input = (float*)malloc(size);
    float *h_output = (float*)malloc(sizeof(float));

    // Initialize array
    for(int i = 0; i < N; i++) {
        h_input[i] = 1.0;
    }

    float *d_input, *d_output;
    cudaMalloc((void**)&d_input, size);
    cudaMalloc((void**)&d_output, sizeof(float));

    cudaMemcpy(d_input, h_input, size, cudaMemcpyHostToDevice);

    // Launch kernel
    reduceSum<<<1, N>>>(d_input, d_output);

    cudaMemcpy(h_output, d_output, sizeof(float), cudaMemcpyDeviceToHost);

    printf("GPU Sum: %f\n", h_output[0]);

    // -------- CPU SUM --------
    float cpu_sum = 0;
    for(int i = 0; i < N; i++) {
        cpu_sum += h_input[i];
    }

    printf("CPU Sum: %f\n", cpu_sum);

    // Cleanup
    cudaFree(d_input); cudaFree(d_output);
    free(h_input); free(h_output);

    return 0;
}

Writing reduction.cu


In [2]:
!nvcc reduction.cu -o reduction
!./reduction

GPU Sum: 1024.000000
CPU Sum: 1024.000000
